# Emergent Introspection — Colab setup & smoke test

Runs the `safety-research/introspection-mechanisms` harness on a free Colab T4. This notebook covers **Stage 0 (environment) and Stage 0.4 (smoke test)** — the steps that need no paid GPU.

**Before you run anything:**
1. **Runtime → Change runtime type → T4 GPU.** Otherwise the model loads on CPU.
2. **Accept the Gemma license.** The smoke-test model `google/gemma-2-2b-it` is gated — accept it on its HuggingFace model page under the *same account* as your `HF_TOKEN`, or the first load will 401.
3. **Keys in Colab Secrets (🔑 sidebar):** add `HF_TOKEN` and `OPENAI_API_KEY`.

You are checking for **"no exceptions"** here — model loads, a vector is built, a generation comes out, the judge returns a label. **Not** for correct detection rates.

> 🔒 Colab disk is ephemeral, semi-public infra. Fine for the 2B smoke test (it produces none of the sensitive artifacts). **Do not mount Drive to cache vectors/activations** — regenerate, never archive.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Expect a Tesla T4 with ~15 GB. If this errors or shows no GPU, fix Runtime → Change runtime type first.

## 2. Clone the harness (public repo, no auth needed)

In [ ]:
import os
%cd /content
if not os.path.exists('introspection-mechanisms'):
    !git clone https://github.com/safety-research/introspection-mechanisms.git
else:
    print('already cloned')
%cd /content/introspection-mechanisms

## 3. Install

`requirements.txt` pins `numpy<2.0` and Colab ships numpy 2.x, so pip will downgrade it and Colab will show a **"Restart runtime"** button. Click it, then re-run cell 2 (`%cd`) before continuing.

The heavy deps (`sae-lens`, `optuna`, `kaleido`, `networkx`) aren't needed for the smoke test or G3. If the full install is slow or fights you, comment the first line and use the slim install instead.

In [ ]:
!pip install -q -r requirements.txt
# Slim alternative (enough for smoke test + G3):
# !pip install -q torch transformers accelerate openai python-dotenv pandas "numpy<2.0" tqdm scikit-learn

## 4. Load your keys into the environment

The harness reads `HF_TOKEN` (via `transformers`) and `OPENAI_API_KEY` (the judge) from **environment variables**. This cell pulls them from Colab Secrets into `os.environ`. Never hardcode keys in the notebook — it's committed to GitHub.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# sanity check without printing the secrets:
print('HF set:', bool(os.environ.get('HF_TOKEN')), '| OpenAI set:', bool(os.environ.get('OPENAI_API_KEY')))

## 5. Smoke test (Stage 0.4)

Real flags. Note: there is **no `gemma3_4b`** in the registry — the smallest in-registry Gemma is `gemma2_2b` (Gemma-**2**). `--layer-fraction 0.6` ≈ L37/62, `--strength` is α.

**Expect** a handful of generations and judge labels written to an output dir, and no exceptions. If the judge call errors, it's the OpenAI key/quota — fix it here where it's free.

In [ ]:
!python experiments/01_concept_injection.py \
    --models gemma2_2b --concepts bread hammer \
    --strength 4.0 --layer-fraction 0.6 --n-trials 2

## 6. G3 — geometry premise check *(next)*

The decisive, inference-only step: *are harmful concept vectors more refusal-aligned than benign ones at the injection layer?* This is mostly hand-written (not run through the harness) — refusal-direction extraction + concept vectors + the cosine sweep across pooling protocols and layers.

**This section will be filled in by the next push.** When it appears here after you reopen the notebook from GitHub, the sync workflow is working.